# Bill Total & Date Extractor – OCR + joblib

## Cell 1 – Install Tesseract and libraries

In [1]:
!apt-get -qq install -y tesseract-ocr tesseract-ocr-sin tesseract-ocr-tam > /dev/null
!pip install -q pytesseract python-dateutil

## Cell 2 – Load training receipts (SROIE)

In [2]:
from datasets import load_dataset
ds = load_dataset("vishu12121/ICDAR2019-SROIE")
train_ds, test_ds = ds["train"], ds["test"]
print(len(train_ds), "train receipts,", len(test_ds), "test receipts")

README.md:   0%|          | 0.00/14.3k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  319MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  191MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/626 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/361 [00:00<?, ? examples/s]

626 train receipts, 361 test receipts


## Cell 3 – OCR, features and rules

In [3]:
import re
import numpy as np
import pandas as pd
import pytesseract
from PIL import Image, ImageOps
from dateutil import parser as dparser

KW = {
    "kw_total": ["total", "amount due", "amount payable", "payable", "nett", "net amount", "මුළු", "එකතුව", "மொத்த"],
    "kw_sub": ["sub total", "subtotal", "sub-total"],
    "kw_cash": ["cash", "tender", "paid", "change", "balance", "refund", "card", "visa", "master"],
    "kw_tax": ["tax", "gst", "vat", "sst", "service"],
    "kw_disc": ["disc", "saving"],
    "kw_round": ["round"],
    "kw_item": ["item", "qty", "quantity"],
}
FEATURES = ["rel_y", "rel_x", "rank", "is_max", "n_same", "n_cands"] + list(KW) + [k + "_prev" for k in KW]

AMOUNT_RE = re.compile(r"\d{1,3}(?:,\d{3})+(?:\.\d{2})?|\d+[.,]\d{2}(?!\d)")
DATE_RE = re.compile(
    r"\d{4}[/.-]\d{1,2}[/.-]\d{1,2}|\d{1,2}[/.-]\d{1,2}[/.-]\d{2,4}"
    r"|\d{1,2}\s?(?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*\s?\d{2,4}", re.I)


def to_amount(s):
    if s is None:
        return None
    s = re.sub(r"[^\d.,]", "", str(s)).strip(".,")
    if not s:
        return None
    if "," in s and "." in s:
        s = s.replace(",", "")
    elif "," in s:
        s = s.replace(",", "") if len(s.split(",")[-1]) == 3 else s.replace(",", ".")
    try:
        return round(float(s), 2)
    except ValueError:
        return None


def to_date(s):
    if not s:
        return None
    s = str(s)
    yf = bool(re.match(r"\d{4}([./-]|\d{4}$)", s.strip()))
    try:
        return dparser.parse(s, dayfirst=not yf, yearfirst=yf, fuzzy=True).date().isoformat()
    except (ValueError, OverflowError):
        return None


def ocr_words(img, lang="eng+sin+tam"):
    g = ImageOps.exif_transpose(img).convert("L")
    if g.width < 1000:
        g = g.resize((g.width * 2, g.height * 2))
    elif g.width > 1800:
        g = g.resize((1600, int(g.height * 1600 / g.width)))
    d = pytesseract.image_to_data(g, lang=lang, config="--psm 6", output_type=pytesseract.Output.DICT)
    words = [{"t": t.strip(), "x": x, "w": w, "yc": y + h / 2, "h": h}
             for t, x, y, w, h in zip(d["text"], d["left"], d["top"], d["width"], d["height"]) if t.strip()]
    return words, g.height


def words_text(words):
    return " ".join(w["t"] for w in sorted(words, key=lambda w: (round(w["yc"] / 10), w["x"])))


def row_text(words, yc, h, shift=0.0):
    target = yc + shift * h
    row = [w for w in words if abs(w["yc"] - target) < 0.6 * max(h, w["h"])]
    return " ".join(w["t"] for w in sorted(row, key=lambda w: w["x"])).lower()


def candidates(words, height):
    cands = []
    for w in words:
        if DATE_RE.search(w["t"]):
            continue
        for m in AMOUNT_RE.findall(w["t"]):
            v = to_amount(m)
            if v and v > 0:
                cands.append({"value": v, "yc": w["yc"], "h": w["h"], "right": w["x"] + w["w"]})
    if not cands:
        return pd.DataFrame()
    vals = np.array([c["value"] for c in cands])
    width = max(w["x"] + w["w"] for w in words)
    rows = []
    for c in cands:
        same = row_text(words, c["yc"], c["h"])
        prev = row_text(words, c["yc"], c["h"], shift=-1.8)
        f = {"value": c["value"], "rel_y": c["yc"] / height, "rel_x": c["right"] / width,
             "rank": (vals > c["value"]).sum() / len(vals), "is_max": float(c["value"] == vals.max()),
             "n_same": float((vals == c["value"]).sum()), "n_cands": float(len(vals))}
        for k, kws in KW.items():
            f[k] = float(any(s in same for s in kws))
            f[k + "_prev"] = float(any(s in prev for s in kws))
        rows.append(f)
    return pd.DataFrame(rows)


def find_date(text):
    for m in DATE_RE.finditer(text):
        d = to_date(m.group(0))
        if d and 2000 <= int(d[:4]) <= 2035:
            return d
    return None


def find_currency(text):
    t = text.lower()
    if re.search(r"(?<![a-z])rs(?![a-z])|lkr|රු|ரூ", t):
        return "LKR"
    if "usd" in t or "$" in t:
        return "USD"
    if re.search(r"(?<![a-z])rm(?![a-z])", t):
        return "MYR"
    return None


def pick_total(words, height, clf):
    df = candidates(words, height)
    if df.empty:
        return None
    p = clf.predict_proba(df[FEATURES])[:, 1]
    return float(df["value"].iloc[p.argmax()])


def read_bill(img, clf):
    words, height = ocr_words(img)
    text = words_text(words)
    return {"total": pick_total(words, height, clf), "date": find_date(text), "currency": find_currency(text)}

## Cell 4 – OCR all receipts (15–30 min)

In [6]:
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

def ocr_dataset(ds):
    def one(i):
        ex = ds[i]
        words, height = ocr_words(ex["image"], lang="eng")
        return {"words": words, "height": height,
                "total": to_amount(ex["entities"]["total"]), "date": to_date(ex["entities"]["date"])}
    with ThreadPoolExecutor(4) as pool:
        return list(tqdm(pool.map(one, range(len(ds))), total=len(ds)))

train_docs = ocr_dataset(train_ds.select(range(75)))
test_docs = ocr_dataset(test_ds.select(range(35)))
print("OCR done:", len(train_docs), "train,", len(test_docs), "test")

  0%|          | 0/75 [00:00<?, ?it/s]

  0%|          | 0/35 [00:00<?, ?it/s]

OCR done: 75 train, 35 test


## Cell 5 – Build table and train

In [7]:
from sklearn.ensemble import RandomForestClassifier

def make_table(docs):
    parts = []
    for i, d in enumerate(docs):
        df = candidates(d["words"], d["height"])
        if df.empty or not d["total"]:
            continue
        df["label"] = ((df["value"] - d["total"]).abs() < 0.005).astype(int)
        df["doc"] = i
        parts.append(df)
    return pd.concat(parts, ignore_index=True)

train_tab = make_table(train_docs)
print(len(train_tab), "candidate rows,", int(train_tab["label"].sum()), "are totals")
display(train_tab.head(10))

clf = RandomForestClassifier(n_estimators=200, max_depth=12, min_samples_leaf=2,
                             class_weight="balanced", random_state=42, n_jobs=-1)
clf.fit(train_tab[FEATURES], train_tab["label"])
print("Training done")

647 candidate rows, 118 are totals


,value,rel_y,rel_x,rank,is_max,n_same,n_cands,kw_total,kw_total_prev,kw_sub,...,kw_tax,kw_tax_prev,kw_disc,kw_disc_prev,kw_round,kw_round_prev,kw_item,kw_item_prev,label,doc
0,9.0,0.596249,0.552809,0.2,0.0,2.0,5.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0
1,6.0,0.596742,0.693258,0.6,0.0,1.0,5.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0
2,9.0,0.596249,0.996629,0.2,0.0,2.0,5.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0
3,1.0,0.638203,0.995506,0.8,0.0,1.0,5.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0
4,9.6,0.702369,0.997753,0.0,1.0,1.0,5.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0,0
5,10.0,0.574452,0.715698,0.7,0.0,2.0,10.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1
6,19.0,0.573954,0.923362,0.5,0.0,1.0,10.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1
7,55.9,0.624004,0.714462,0.3,0.0,2.0,10.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1
8,55.9,0.623506,0.923362,0.3,0.0,2.0,10.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1
9,10.0,0.648655,0.667491,0.7,0.0,2.0,10.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1


Training done


## Cell 6 – Check accuracy

In [8]:
n = found = ok_total = ok_date = 0
for d in test_docs:
    if not d["total"]:
        continue
    n += 1
    cands = candidates(d["words"], d["height"])
    found += (not cands.empty) and bool(((cands["value"] - d["total"]).abs() < 0.005).any())
    ok_total += pick_total(d["words"], d["height"], clf) == d["total"]
    ok_date += find_date(words_text(d["words"])) == d["date"]

print("Test receipts:", n)
print("Total correct:", round(ok_total / n * 100, 1), "%")
print("Date correct:", round(ok_date / n * 100, 1), "%")
print("OCR read the total at all:", round(found / n * 100, 1), "%  (best possible for total)")

Test receipts: 35
Total correct: 80.0 %
Date correct: 77.1 %
OCR read the total at all: 94.3 %  (best possible for total)


## Cell 7 – Save and download bill_model.joblib

In [9]:
import joblib
from google.colab import files

joblib.dump({"model": clf, "features": FEATURES}, "bill_model.joblib")
files.download("bill_model.joblib")

import sklearn
print("scikit-learn version:", sklearn.__version__)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

scikit-learn version: 1.6.1


## Cell 8 – Test on your own bills

In [ ]:
from google.colab import files
uploaded = files.upload()
rows = [{"file": name, **read_bill(Image.open(name), clf)} for name in uploaded]
display(pd.DataFrame(rows))